# Cap10 Metric Replay Handoff Audit V1

- **Baselines**: canonical `v19reg_v22_budget_usage_bull_guard_v1`; source diagnostic `cap10_wf2020_bull_participation_diagnostic_v1`; metric replay `cap10_three_fold_checkpoint_metric_replay_v2`; reference `soxlbacktest_qlib_buysell_reference`.
- **Role**: diagnose.
- **Target window**: `wf_2020`, `wf_2022`, `wf_2023_2024`.
- **Changed layer**: model_selection / metric_reconciliation / lineage.
- **Hypothesis**: `cap10_three_fold_checkpoint_metric_replay_v2` already performed the requested same-source qlib metric rerun, so the mixed-metric source diagnostic should be closed and the frontier should advance to the remaining fold failure.
- **Acceptance criteria**: the replay notebook uses `backtest_profile` outputs `bnh_return` and `relative_outperformance`, all required folds are present, and remaining failure is reported from replay metrics.
- **Failure mode**: if the replay notebook does not expose same-source benchmark fields or misses a fold, continue blocking training on metric reconciliation.


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'SPEC.md').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.metric_replay_handoff import write_metric_replay_handoff_audit

SOURCE_CARD = ROOT / 'runtime/model_selection/card/cap10_wf2020_bull_participation_diagnostic_v1'
REPLAY_CARD = ROOT / 'runtime/model_selection/card/cap10_three_fold_checkpoint_metric_replay_v2'
OUTPUT_DIR = ROOT / 'runtime/model_selection/card/cap10_metric_replay_handoff_audit_v1'

audit = write_metric_replay_handoff_audit(
    source_card_dir=SOURCE_CARD,
    replay_card_dir=REPLAY_CARD,
    output_dir=OUTPUT_DIR,
)
audit.summary


{'experiment': 'cap10_metric_replay_handoff_audit_v1',
 'created_at': '2026-06-13T19:00:53.493058+00:00',
 'decision': 'metric_replay_handoff_completed_no_promotion',
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'training_allowed_from_this_card': False,
 'role': 'diagnose',
 'changed_layer': ['model_selection', 'metric_reconciliation', 'lineage'],
 'source_experiment': 'cap10_wf2020_bull_participation_diagnostic_v1',
 'direct_comparison_baseline': 'cap10_three_fold_checkpoint_metric_replay_v2',
 'metric_replay_experiment': 'cap10_three_fold_checkpoint_metric_replay_v2',
 'source_requires_metric_replay': True,
 'metric_replay_source_fields_present': True,
 'metric_replay_fold_coverage': True,
 'metric_replay_handoff_completed': True,
 'required_folds': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
 'folds_evaluated': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
 'missing_folds': [],
 'failed_folds': ['wf_2020'],
 'selected_by_fold': {'wf_2020': {'selected_candidate': 'bc_initia

In [2]:
pd.read_csv(OUTPUT_DIR / 'fold_metric_replay_handoff.csv')


,fold,selected_candidate,selected_candidate_gate,selected_max_dd,selected_relative_return
0,wf_2020,bc_initialization,False,14.984701,-292.753021
1,wf_2022,bc_initialization,True,18.802284,65.864820
2,wf_2023_2024,ppo_retained_iter_4,True,8.955769,35.566229


## Conclusion

- **Decision**: `metric_replay_handoff_completed_no_promotion`.
- **Promotion / retention**: no actor promotion and no actor retention from this audit.
- **Key evidence**: `cap10_three_fold_checkpoint_metric_replay_v2` uses `backtest_profile` output fields `bnh_return` and `relative_outperformance`, and covers `wf_2020`, `wf_2022`, and `wf_2023_2024`.
- **Remaining gate failure**: after the metric replay, `wf_2020` still fails relative return (`-292.75pp`) while MaxDD is below the `<20%` gate (`14.98%`). `wf_2022` and `wf_2023_2024` pass the replay fold gate.
- **Next action**: inspect the failed `wf_2020` fold for reward drift or drawdown/re-entry objective changes before full retention training.
- **Artifacts**: `summary.json`, `selection.json`, `fold_metric_replay_handoff.csv`, `conclusion.md`.
